# NUMPY MASTERY — CHAPTERS 27–29

```
NUMPY MASTERY
Current Chapters: 27 (Numerical Stability), 28 (Memory Layout and Strides),
                  29 (Vectorization)
Progress:         29 / 75 chapters
Completed:        Ch1-26
Remaining:        46 chapters
```

This trio is "how NumPy really behaves under the hood": why floating-point math isn't exact
(27), how memory layout actually determines performance (28), and mature judgment about when
vectorization helps vs hurts (29) — direct preparation for Chapter 30's formal performance
engineering.


In [ ]:
import numpy as np
print("NumPy version:", np.__version__)


---
# CHAPTER 27 — Numerical Stability

## Concept
Computers represent real numbers with **finite precision** (IEEE 754 floating point). This
means arithmetic that's exact in mathematics (`0.1 + 0.2 == 0.3`) can be inexact in code, and
certain sequences of operations (subtracting nearly-equal large numbers, repeated
multiplication of many small/large values) can amplify tiny representation errors into large,
meaningful ones.

## Why it matters
Every loss function, every gradient computation, every matrix inversion in ML is built from
millions of floating-point operations. A numerically unstable formulation of a *mathematically
correct* formula can silently produce wrong or `NaN` results, especially at scale.


In [ ]:
print("0.1 + 0.2 == 0.3 ?", 0.1 + 0.2 == 0.3)
print("0.1 + 0.2 actual value:", repr(0.1 + 0.2))
print()
print("np.isclose(0.1 + 0.2, 0.3)         :", np.isclose(0.1 + 0.2, 0.3))
print("np.allclose([0.1+0.2, 1.0], [0.3, 1.0 + 1e-9]) :",
      np.allclose([0.1 + 0.2, 1.0], [0.3, 1.0 + 1e-9]))


### Expected Output
```
0.1 + 0.2 == 0.3 ? False
0.1 + 0.2 actual value: 0.30000000000000004

np.isclose(0.1 + 0.2, 0.3)         : True
np.allclose([0.1+0.2, 1.0], [0.3, 1.0 + 1e-9]) : True
```

### Why `0.1 + 0.2 != 0.3` — the mechanism, not just the fact
`0.1` and `0.2` have no exact representation in binary floating point (just like `1/3` has no
exact finite decimal representation) — they're stored as the *closest representable* double,
which carries a tiny rounding error. Adding two such approximations compounds the error
slightly, landing just above `0.3`'s own closest representable value. This is fundamental to
**all** IEEE 754 floating-point systems (not a NumPy quirk) and is why you must **never**
compare floats with `==`; always use `np.isclose()` (element-wise) or `np.allclose()`
(whole-array boolean) with sensible `rtol`/`atol` tolerances.

### Catastrophic cancellation — a more dangerous, less obvious failure mode


In [ ]:
# Subtracting two nearly-equal large numbers destroys precision
a = 1.00000001
b = 1.00000000
naive_diff = a - b
print("naive a - b            :", naive_diff, "  (mathematically should be exactly 1e-8)")
print("relative error          :", abs(naive_diff - 1e-8) / 1e-8)

# A classic case: computing variance via E[X^2] - E[X]^2 (the "naive" formula) vs
# directly computing E[(X - mean)^2] (the "stable" formula), for data with a huge offset
rng = np.random.default_rng(0)
data = rng.normal(loc=1e8, scale=1.0, size=10_000)   # small variance, HUGE mean offset

naive_var = np.mean(data**2) - np.mean(data)**2      # textbook formula, numerically DANGEROUS
stable_var = np.mean((data - np.mean(data))**2)      # direct formula, numerically SAFE
true_var_via_numpy = np.var(data)                     # NumPy's own internal implementation is stable

print(f"\nnaive formula   variance: {naive_var}")
print(f"stable formula  variance: {stable_var}")
print(f"np.var (ground truth)   : {true_var_via_numpy}")


### Expected Output (verified on NumPy 2.4.4)
```
naive a - b            : 9.99999993922529e-09   (mathematically should be exactly 1e-8)
relative error          : 6.0774709918447105e-09   <- small here, but this is JUST 2 numbers

naive formula   variance: 2.0                      <- wrong! true variance is ~1.0, off by 2x
stable formula  variance: 0.9961574235021667
np.var (ground truth)   : 0.9961574235021667
```

### Reading the catastrophic cancellation example
The "naive" variance formula `E[X²] - E[X]²` is **algebraically identical** to the stable
formula — they're the same mathematics. But when `data` has a huge mean (`~1e8`) and small
variance (`~1`), `X²` values are enormous (`~1e16`) while their true differences are tiny
(`~1`). Subtracting two enormous, nearly-equal floating-point numbers (`mean(X²)` and
`mean(X)²`, both ≈ `1e16`) destroys almost all significant digits of precision in the tiny
difference that remains — above, the naive formula lands on `2.0` instead of the true `~0.996`,
roughly double the correct value, purely from rounding error in that subtraction. (Depending on
the exact data and offset magnitude, this same mechanism can push the naive result to a wildly
wrong value or even **negative** — mathematically impossible for a true variance.) This is
exactly why `np.var()`'s internal implementation (and any competent numerical library) uses the
direct, mean-centered formula, never the naive `E[X²]-E[X]²` shortcut, despite the shortcut
being algebraically "equivalent" on paper.

### Stable softmax — the ML-specific instance of this same lesson (previewed here, fully
### implemented in Chapter 45)


In [ ]:
def naive_softmax(x):
    exps = np.exp(x)
    return exps / exps.sum()

def stable_softmax(x):
    shifted = x - np.max(x)     # subtract the max BEFORE exponentiating
    exps = np.exp(shifted)
    return exps / exps.sum()

large_logits = np.array([1000., 1001., 1002.])   # realistic-scale raw model outputs

with np.errstate(over='ignore', invalid='ignore'):   # let the naive version fail visibly, don't crash the notebook
    naive_result = naive_softmax(large_logits)

stable_result = stable_softmax(large_logits)

print("naive_softmax(large_logits) :", naive_result, "  <- overflowed to inf, then inf/inf = nan!")
print("stable_softmax(large_logits):", stable_result, "  <- correct, well-defined probabilities")
print("stable result sums to 1?    :", np.isclose(stable_result.sum(), 1.0))


### Expected Output
```
naive_softmax(large_logits) : [nan nan nan]   <- overflowed to inf, then inf/inf = nan!
stable_softmax(large_logits): [0.09003057 0.24472847 0.66524096]   <- correct, well-defined probabilities
stable result sums to 1?    : True
```

### Why subtracting the max fixes it, mathematically
`softmax(x)_i = exp(x_i) / sum(exp(x_j))`. Subtracting a constant `c` from every `x_i` before
exponentiating leaves the mathematical result **exactly unchanged** (it cancels out in the
ratio: `exp(x_i - c) / sum(exp(x_j - c)) = exp(x_i)/exp(c) / (sum(exp(x_j))/exp(c)) =
exp(x_i)/sum(exp(x_j))`), but choosing `c = max(x)` guarantees every exponent argument is `≤ 0`,
so `exp()` never overflows (its output is always in `(0, 1]`). This is a textbook example of
"reformulate a mathematically-equivalent expression to be numerically stable" — the exact
mindset this whole chapter is building.

### Common Mistakes
- Comparing floats with `==` anywhere in a pipeline (data validation, test assertions, cache
  keys) — always `np.isclose`/`np.allclose`.
- Using the naive `E[X²]-E[X]²` variance/covariance shortcut on data with large offsets.
- Exponentiating large raw values (logits, unscaled features) without a stabilizing shift —
  applies to `exp`, and by extension anything built on it (softmax, sigmoid at extreme inputs).

### Interview Insight
> "Why might `np.var()` never use the `E[X²] - E[X]²` formula, even though it's mathematically
> equivalent to the direct approach?"
For data with a large mean relative to its variance, `E[X²]` and `E[X]²` are both huge and
nearly equal — subtracting them causes catastrophic cancellation, potentially even producing a
negative "variance." The direct, mean-centered formula avoids ever forming those large
intermediate values, preserving precision.


---
# CHAPTER 28 — Memory Layout and Strides

## Concept
An array's data lives in one flat, contiguous block of memory. `strides` (Chapter 3 preview,
now fully explored) describes how many **bytes** to step to move one position along each axis.
The **order** in which elements are actually laid out in that flat block — row-major ("C
order", NumPy's default) or column-major ("Fortran order") — has real, measurable performance
consequences because of CPU cache behavior.

## Why it matters
Iterating over memory in the order it's physically laid out is dramatically faster than jumping
around, because modern CPUs fetch memory in cache-line-sized chunks (typically 64 bytes) — if
your access pattern matches physical layout, each cache line fetch serves many subsequent
accesses; if it doesn't, nearly every access is a cache miss.


In [ ]:
arr_c = np.arange(24, dtype=np.float64).reshape(4, 6)          # C order (default)
arr_f = np.asfortranarray(arr_c)                                # same data, F order

print("arr_c.flags['C_CONTIGUOUS']:", arr_c.flags['C_CONTIGUOUS'], " strides:", arr_c.strides)
print("arr_f.flags['F_CONTIGUOUS']:", arr_f.flags['F_CONTIGUOUS'], " strides:", arr_f.strides)
print()
print("Same values?", np.array_equal(arr_c, arr_f))
print("Same memory ORDER? (they are NOT) -- arr_c strides vs arr_f strides differ:",
      arr_c.strides != arr_f.strides)


### Expected Output
```
arr_c.flags['C_CONTIGUOUS']: True  strides: (48, 8)
arr_f.flags['F_CONTIGUOUS']: True  strides: (8, 32)

Same values? True
Same memory ORDER? (they are NOT) -- arr_c strides vs arr_f strides differ: True
```

### Reading these strides
- `arr_c` (C order, the NumPy default): strides `(48, 8)` for shape `(4, 6)`, `float64`
  (8 bytes/element) — moving to the next **column** (last axis) costs only 8 bytes (adjacent in
  memory); moving to the next **row** costs 48 bytes (= 6 elements × 8 bytes, a full row's
  worth). This means **rows are stored contiguously** — walking across a row is a cache-friendly
  sequential memory scan.
- `arr_f` (Fortran order): strides `(8, 32)` — the opposite: moving down a **column** costs only
  8 bytes; **columns are stored contiguously** instead.

### Why this matters for performance — a measurable demonstration


In [ ]:
import time

n = 4000
big_c = np.random.default_rng(0).normal(size=(n, n))            # C-contiguous (NumPy default)
big_f = np.asfortranarray(big_c)                                  # same data, F-contiguous

def sum_by_rows(arr):
    total = 0.0
    for i in range(arr.shape[0]):
        total += arr[i, :].sum()     # accesses one ROW at a time
    return total

t0 = time.perf_counter()
sum_by_rows(big_c)   # row access on C-order array -> matches physical layout -> fast
t1 = time.perf_counter()

sum_by_rows(big_f)   # row access on F-order array -> fights physical layout -> slower
t2 = time.perf_counter()

print(f"Row-wise summation on C-contiguous array: {t1 - t0:.4f} s")
print(f"Row-wise summation on F-contiguous array: {t2 - t1:.4f} s")
print(f"Slowdown factor: {(t2 - t1) / (t1 - t0):.2f}x")


### Expected Output (verified on NumPy 2.4.4 — exact ratio depends on hardware/cache size)
```
Row-wise summation on C-contiguous array: 0.0243 s
Row-wise summation on F-contiguous array: 0.1792 s
Slowdown factor: 7.37x
```

### The general principle
**Access memory in the order it's physically laid out.** For NumPy's default C-order arrays,
that means: prefer operations that scan along the *last* axis fastest (row-by-row for 2D).
Whenever you have a choice (e.g., choosing whether to store `(features, samples)` or `(samples,
features)`, or which axis to iterate), align your dominant access pattern with the array's
actual memory order.

### `np.ascontiguousarray()` — forcing C order when it matters
Operations like slicing with a step (`arr[::2]`), or transposing, can produce a **non-contiguous
view** — mathematically correct, but potentially slow for subsequent heavy computation (some
BLAS/LAPACK-backed operations silently copy internally to a contiguous layout before
proceeding, which is itself a hidden performance cost you should know to look for).
```python
X_transposed_view = X.T                       # a view, but often non-contiguous
X_fast = np.ascontiguousarray(X_transposed_view)  # explicit copy into C-contiguous layout,
                                                     # pay the cost ONCE if reused many times
```

### Common Mistake
Repeatedly performing heavy numerical operations on a non-contiguous array (e.g., inside a
training loop) without realizing each call may be silently re-copying to a contiguous layout
internally — `ascontiguousarray()` once, up front, can be a meaningful optimization if the array
is reused many times.

### Interview Insight
> "You have a large 2D array and need to repeatedly sum along its rows in a hot loop. Does the
> array's memory layout matter, and why?"
Yes — for a C-contiguous array (NumPy's default), rows are stored as contiguous memory blocks,
so row-wise access is cache-friendly (sequential memory scan). If the array happens to be
F-contiguous (e.g., after certain operations), the same row-wise access pattern jumps around
memory non-sequentially, causing many more cache misses and measurably slower performance for
the identical mathematical operation.


---
# CHAPTER 29 — Vectorization: Professional Judgment, Not a Blind Rule

## Concept
Vectorization replaces explicit Python-level loops with array-wide operations dispatched to
compiled C code (Chapter 1, 12). It's usually a large win — but "usually" is not "always," and
professional NumPy usage means knowing the exceptions, not reciting a slogan.

## When vectorization is NOT automatically the right call

1. **Very small arrays.** The fixed overhead of dispatching a NumPy operation (type checking,
   broadcasting resolution, ufunc dispatch) can exceed the cost of a tiny Python loop when there
   are only a handful of elements.
2. **Operations that create large temporary arrays.** A "vectorized" expression like `(a - b)
   ** 2` on huge arrays allocates a full-size temporary for `(a - b)` before squaring — for
   memory-constrained situations, a carefully chunked loop (or `out=` parameters, Chapter 12)
   can use dramatically less peak memory even though it's "less vectorized."
3. **Genuinely sequential dependencies.** Some computations (e.g., a recurrence where step `n`
   depends on the *actual computed output* of step `n-1`, not just the input data) cannot be
   vectorized at all without restructuring the algorithm — `np.cumsum`-style operations are
   vectorized internally by NumPy in C, but a *general* Python recurrence usually can't be
   vectorized by simply rewriting `for` as array ops.


In [ ]:
import timeit

# --- Case 1: tiny arrays -- loop overhead can beat vectorization overhead ---
tiny = [1, 2, 3]
tiny_arr = np.array(tiny)

def tiny_loop():
    return [x * 2 for x in tiny]

def tiny_vectorized():
    return tiny_arr * 2

t_loop = timeit.timeit(tiny_loop, number=100_000)
t_vec = timeit.timeit(tiny_vectorized, number=100_000)
print(f"Tiny array (n=3): list comp = {t_loop:.4f}s, vectorized = {t_vec:.4f}s")
print(f"  -> {'vectorized wins' if t_vec < t_loop else 'plain Python wins!'} for this tiny size")


### Expected Output (verified on NumPy 2.4.4 — the direction of the result is the point, not exact timings)
```
Tiny array (n=3): list comp = 0.0185s, vectorized = 0.0941s
  -> plain Python wins! for this tiny size
```
For an array this small, calling into NumPy costs more (argument parsing, dispatch, output
array allocation) than the actual arithmetic does in either approach — the "vectorization is
always faster" slogan breaks down completely at small scale. This crossover point (where NumPy
starts winning) is typically somewhere in the hundreds-to-low-thousands of elements, depending
on the operation and hardware — worth knowing exists, not worth memorizing an exact number for.


In [ ]:
# --- Case 2: memory-aware chunking vs one giant vectorized temp array ---
n = 20_000_000
a = np.random.default_rng(0).normal(size=n).astype(np.float32)
b = np.random.default_rng(1).normal(size=n).astype(np.float32)

print(f"a.nbytes = {a.nbytes / 1e6:.1f} MB   b.nbytes = {b.nbytes / 1e6:.1f} MB")

# fully vectorized: (a - b) creates a full n-sized temporary array BEFORE squaring
def full_vectorized():
    return np.sum((a - b) ** 2)

# chunked: process in blocks, bounded peak memory, still fully vectorized WITHIN each chunk
def chunked(chunk_size=1_000_000):
    total = 0.0
    for start in range(0, n, chunk_size):
        end = min(start + chunk_size, n)
        diff = a[start:end] - b[start:end]
        total += np.sum(diff ** 2)
    return total

result_full = full_vectorized()
result_chunked = chunked()
print("Results match?", np.isclose(result_full, result_chunked))
print("(The chunked version uses a small, BOUNDED amount of extra memory per iteration,")
print(" instead of one large 80MB extra temporary array all at once for the full array.)")


### Expected Output
```
a.nbytes = 80.0 MB   b.nbytes = 80.0 MB
Results match? True
(The chunked version uses a small, BOUNDED amount of extra memory per iteration,
 instead of one large 80MB extra temporary array all at once for the full array.)
```

### Why this matters for real memory-constrained pipelines
`(a - b) ** 2` on two 80MB `float32` arrays allocates roughly another 80MB temporary for `(a -
b)`, then another ~80MB for the squared result, before `sum()` collapses it all — a fully
"vectorized" one-liner can transiently use several times the memory of the inputs themselves.
For datasets that nearly fill available RAM, this is a common source of `MemoryError` that a
chunked approach (still vectorized *within* each chunk — the best of both worlds) avoids. This
is the direct bridge into Chapter 31 (Memory Optimization) and Chapter 32 (Memory-Mapped
Arrays).

### The professional mental model
```
Is the array large enough that per-call overhead is negligible?         -> vectorize
Does the intermediate memory footprint fit comfortably in available RAM? -> vectorize fully
Otherwise                                                                 -> chunk, still
                                                                              vectorizing WITHIN
                                                                              each chunk
Is there a genuine sequential/recursive dependency on computed outputs?   -> vectorization may
                                                                              not apply at all;
                                                                              look for a
                                                                              specialized
                                                                              NumPy function
                                                                              (cumsum, etc.) or
                                                                              accept a loop
```

### Interview Insight
> "Is vectorization always the right choice in NumPy code?"
No — for very small arrays, per-call dispatch overhead can make a plain Python loop faster; for
very large arrays, a single fully-vectorized expression can create large temporary arrays that
exceed available memory, where a chunked-but-still-locally-vectorized approach is more
appropriate; and for genuinely sequential computations, no straightforward vectorization
exists at all. Professional judgment means measuring and reasoning about the specific case, not
reflexively rewriting every loop as an array expression.


---
## Chapters 27–29 Summary

- **Numerical stability**: floating point is inherently approximate — never compare with `==`,
  always `isclose`/`allclose`. Catastrophic cancellation (subtracting nearly-equal large
  numbers, as in the naive variance formula) can produce dramatically wrong or even
  impossible results (negative variance) despite the formula being "mathematically
  equivalent" on paper. Stable softmax (subtract the max before exponentiating) is the
  canonical ML instance of "reformulate for stability without changing the math."
- **Memory layout/strides**: C order (NumPy default) stores rows contiguously; F order stores
  columns contiguously. Matching your access pattern to physical layout is a real, measurable
  performance factor via CPU cache behavior — not a micro-optimization myth.
- **Vectorization**: usually the right call, but not unconditionally — tiny arrays can make loop
  overhead cheaper than NumPy dispatch overhead; large arrays can make a single vectorized
  expression's temporary arrays exceed available memory, where chunked-but-still-vectorized
  code is the professional middle ground.

---

## Mastery Exercises

**Exercise 1:** Explain, using the "cancellation" concept, why computing standard deviation as
`sqrt(mean(x**2) - mean(x)**2)` is a numerically risky implementation choice for real-world
data streams that might have a large constant offset (e.g., temperature in Kelvin instead of
Celsius).

**Exercise 2 (Prediction):** For a `(1000, 1000)` C-contiguous array, would you expect
`arr.sum(axis=1)` (summing along rows) or `arr.sum(axis=0)` (summing along columns) to be
faster, and why, based on strides/memory layout?

**Coding Challenge:** Implement `stable_log_sum_exp(x)` — a numerically stable version of
`log(sum(exp(x)))` (a very common building block in ML loss functions) using the same
"subtract the max" trick as stable softmax. Verify it matches the naive version for small
inputs, and that the naive version fails (produces `inf`/`nan`) for large inputs where your
stable version still works.

**Debugging Challenge:**
```python
def process_huge_dataset(a, b):
    return np.sqrt((a - b) ** 2).mean()

# Called on two 500,000,000-element float64 arrays, this raises MemoryError on a
# 16GB-RAM machine, even though `a` and `b` together are only ~8GB.
```
Explain, referencing this chapter's memory-temporary-array discussion, why this can exceed
available RAM despite the inputs "fitting," and sketch (in words, not full code) a chunked fix.

**Interview Questions:**
1. Why is `0.1 + 0.2 == 0.3` `False` in NumPy/Python/virtually every language using IEEE 754
   floats — is this a bug?
2. Explain catastrophic cancellation with a concrete example relevant to ML/statistics.
3. Why does memory layout (C vs F order) affect performance for mathematically identical
   operations?
4. Give a concrete scenario where a NumPy "vectorized" one-liner would be the *wrong* choice,
   and explain the alternative.

**Quiz:**
- Q1: True/False — `np.allclose()` is always safe to use with its default tolerances,
  regardless of the numeric scale of your data.
- Q2: In stable softmax, what value is subtracted from the input before exponentiating, and
  why does this not change the mathematical result?
- Q3: For a C-contiguous 2D array, which axis is stored contiguously in memory — rows or
  columns?

---

